# Package Installation

Run this cell first, then the UI cells below.

**Binder (recommended for the full widget UI):**
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Vincenzoos/IPSAE-notebook/main?urlpath=lab/tree/ipsae_eval.ipynb)

**Google Colab (quick try; tall UI may clip):**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vincenzoos/IPSAE-notebook/blob/main/ipsae_eval.ipynb)

**Tips**
- Binder: deps come from `requirements.txt`; first launch can take a few minutes to build.
- Colab: if you see old `/storage/viet/...` paths, disconnect/delete the runtime and reopen from GitHub.
- Single Model: use the structure/PAE upload widgets.
- Bulk zip (Binder): upload the zip via the **left file browser**, paste the path into **Zip path**, click **Extract zip**. The widget uploader only works for tiny zips (~8 MB); a stuck `(1)` means use the file-browser path instead.
- Downloads: use the zip helper cell, then download from the file browser.


In [ ]:
# Setup for local Jupyter, Binder, or Google Colab
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output as colab_output

    colab_output.enable_custom_widget_manager()
    REPO_URL = "https://github.com/Vincenzoos/IPSAE-notebook.git"
    REPO_DIR = Path("/content/IPSAE-notebook")
    if not (REPO_DIR / "ipsae.py").exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

# On Binder/local the repo is already the working directory; pip is a no-op if satisfied.
%pip install -q -r requirements.txt
import ipywidgets


def _find_additions() -> Path:
    """Locate additions/ from cwd"""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "additions"
        if (candidate).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find additions/ in project root"
    )


ADDITIONS = _find_additions()
if str(ADDITIONS) not in sys.path:
    sys.path.insert(0, str(ADDITIONS))

print(f"ipywidgets {ipywidgets.__version__} OK")
print(f"cwd: {Path.cwd().resolve()}")
print(f"additions: {ADDITIONS}")
print("Next: run the ipSAE Evaluation cell below.")


Note: you may need to restart the kernel to use updated packages.
ipywidgets 8.1.8 OK
cwd: /storage/viet/FreeBindCraft/dependencies/IPSAE
additions: /storage/viet/FreeBindCraft/dependencies/IPSAE/additions
Next: run the ipSAE Evaluation cell below.


# ipSAE Evaluation

Run single-model or bulk AlphaFold Server evaluations. Upload structure/PAE files (or a zip for bulk), or type paths already on the machine.


In [ ]:
# Requires to run the Package Installation cell above.
from ipsae_eval_ui import launch_ipsae_eval_ui

launch_ipsae_eval_ui()


# ipSAE Comparison CSV

Summarize collected ipSAE outputs (for example a `bulk_ipsae_evals_YYYYMMDD_HHMMSS/` folder) into a ranked comparison table / CSV.


In [3]:
# Requires the Package Installation / setup cell above (adds additions/ to sys.path).
import importlib

import ipsae_comparison_ui

importlib.reload(ipsae_comparison_ui)
ipsae_comparison_ui.launch_ipsae_comparison_ui()


In [4]:
# Zip a folder for download (useful on Colab / remote notebooks).
# Requires the Package Installation / setup cell above (adds additions/ to sys.path).
import shutil
import subprocess

import ipywidgets as widgets
from IPython.display import clear_output, display

from folder_picker import folder_value, make_folder_picker
from paths import ROOT as REPO_ROOT

folder_dd, refresh_btn, folder_row = make_folder_picker(description="Folder", dropdown_width="520px")
zip_btn = widgets.Button(description="Zip folder", button_style="success", icon="file-archive")
status = widgets.HTML(value=f'<span style="color:#57606a">Select a folder under {REPO_ROOT}, then Zip.</span>')
log = widgets.Output()


def _zip_folder(_):
    name = folder_value(folder_dd)
    src = REPO_ROOT / name
    with log:
        clear_output(wait=True)
        if not name or not src.is_dir():
            status.value = '<span style="color:#cf222e">Folder not found.</span>'
            return
        if shutil.which("zip") is None:
            subprocess.run(["apt-get", "update"], check=False)
            subprocess.run(["apt-get", "install", "-y", "zip", "unzip"], check=True)
        print(f"zip -r {name}.zip {name}")
        result = subprocess.run(["zip", "-r", f"{name}.zip", name], cwd=str(REPO_ROOT))
        if result.returncode == 0:
            status.value = f'<span style="color:#1a7f37;font-weight:600">Created {REPO_ROOT / (name + ".zip")}</span>'
        else:
            status.value = '<span style="color:#cf222e">zip failed (see log).</span>'


zip_btn.on_click(_zip_folder)
display(widgets.VBox([folder_row, zip_btn, status, log]))
